# Lab 3.3 · The bias of a customer-randomised dispatch test

Day 3 · Session 3 (Interference in marketplaces) · 25 min.

Joud Al-Fahad, who runs Sufrah's couriers, tested a new dispatch rule (T-301) on half the customers: their
orders go to the next free courier first, and when a whole city-hour runs the rule, batching nearby orders adds
capacity. The test showed treated customers getting their food much faster. In the marketplace simulator you
measure that naive effect, then the effect of running the rule everywhere, and the bias between them.

**AI policy: AI-tutor.** An assistant may explain the queueing idea; the measurements are yours.

**Reference state.** `U.dispatch_sim()`: six cities, 14 operating hours a day for 14 days, couriers shared by
treated and control customers. Nothing from another lab.

**Tasks**
- [ ] 1 · Run the customer-randomised test (4')
- [ ] 2 · The naive effect on delivery time and conversion (5')
- [ ] 3 · The global effect: all treated against none (6')
- [ ] 4 · The bias, over 30 runs (6')
- [ ] 5 · Write the first half of the memo (4')
- [ ] ✱ Stretch · Treat 10% of customers instead of 50% and see the bias change

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/online-experiments.git"
    if not Path("/content/online-experiments").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/online-experiments"], check=True)
    os.chdir("/content/online-experiments/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("sufrah.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import sufrah as U

## 1 · The customer-randomised test (4')

One row per city, day, operating hour and arm: sessions, orders, mean delivery minutes.

In [ ]:
test = U.dispatch_sim("customer", seed=101)
print(test.head())
print(test.groupby("arm")[["sessions", "orders"]].sum())

## 2 · The naive effect (5')

✏️ **Exercise** · Write `summary(d)`: conversion (orders over sessions) and mean delivery minutes weighted by
orders, for a frame of rows. Then the naive effect is treatment minus control.

In [ ]:
def summary(d: pd.DataFrame) -> pd.Series:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
def naive_effect(d: pd.DataFrame) -> pd.Series:
    return summary(d[d.arm == "treatment"]) - summary(d[d.arm == "control"])


try:
    naive = naive_effect(test)
except NotImplementedError:
    naive = pd.Series(U.reference("d3_3_naive"))
U.keep("d3_3_naive", naive.to_dict())
print("naive effect (treatment minus control):", naive.round(4).to_dict())
U.check(naive["delivery_min"] < -5, "treated customers look much faster")

## 3 · The global effect (6')

The question the business asks is: what happens if **everyone** gets the new rule? Run the simulator with all
orders treated and with none, and compare.

✏️ **Exercise** · Write `global_effect(seed)`: summary of `assign="all"` minus summary of `assign="none"`.

In [ ]:
def global_effect(seed: int) -> pd.Series:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    glob = global_effect(201)
except NotImplementedError:
    glob = pd.Series(U.reference("d3_3_global"))
U.keep("d3_3_global", glob.to_dict())
print("global effect (all minus none):", glob.round(4).to_dict())
U.check(naive["delivery_min"] < glob["delivery_min"] < 0, "the global gain is real but far smaller")

🔍 **Look at** why. A priority rule does not create couriers: it moves the wait from control orders to treated
ones. In a 50/50 test, the control arm is slower than it would be without the test, and the treated arm faster
than it would be if everyone had the rule. Only the batching gain is real.

## 4 · The bias, over 30 runs (6')

One run is noisy. Repeat both designs 30 times with different seeds and report the mean naive effect, the
mean global effect and their difference: the bias of the customer-randomised test.

✏️ **Exercise** · Fill `bias_table(runs)`.

In [ ]:
RUNS = U.pick(30, 10)


def bias_table(runs: int) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    bias = bias_table(RUNS)
except NotImplementedError:
    bias = U.reference("d3_3_bias")
U.keep("d3_3_bias", bias)
print(bias.round(4))
ratio = bias.loc["delivery_min", "naive"] / bias.loc["delivery_min", "global"]
print(f"the naive test overstates the delivery-time gain {ratio:.1f} times")
U.check(ratio > 3, "the naive estimate is several times the real effect")

## 5 · The memo, first half (4')

✏️ **Exercise** · Write `memo_part1()`: three or four sentences for Joud. What the test claimed, what running the
rule everywhere would really do, the size of the bias, and why. Lab 3.4 writes the second half: the design.

In [ ]:
def memo_part1() -> str:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    MEMO1 = memo_part1()
except NotImplementedError:
    MEMO1 = U.reference("d3_3_memo")
U.keep("d3_3_memo", MEMO1)
U.save("T301_memo_part1.md", "# Dispatch test T-301: what the customer-randomised test measured\n\n" + MEMO1 + "\n")
print(MEMO1)
U.check(len(MEMO1) > 150, "the first half of the memo is written")

## ✱ Stretch · Treat 10%

Run `U.dispatch_sim("customer", p_treat=0.1)` and compare the naive effect with the 50/50 test. It changes with
the split, and stays far from the global effect: the bias of a customer-randomised test depends on the split
itself. What does that say about reading a 5% ramp of a marketplace feature as its effect?

## Checkpoint

1. Which assumption does the dispatch test break? <details><summary>Answer</summary>No interference (SUTVA): a
   control customer's outcome depends on which other customers are treated.</details>
2. Why does the bias point toward a bigger gain? <details><summary>Answer</summary>The treated arm gains what the
   control arm loses; their difference counts the transfer twice and the real gain once.</details>
3. Name two other Sufrah tests that share a resource. <details><summary>Answer</summary>Restaurant ranking
   (kitchen capacity), courier incentives (courier time), promotions with a limited budget.</details>

**Carry forward.** Lab 3.4 designs the test that can measure this change.